[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Context Managers and Iterators


## What you will be able to do

Write a class that works with `with`, so its cleanup runs even when the block fails, and a class
that works with `for`, so it can be looped over, summed, sorted and searched like a list.


## The idea

### The problem

Every file in this library has been opened with `with open(...) as f:`, and every list, file and
CSV reader has been walked with `for`. Both statements have only ever been used on objects somebody
else built.

Consider what each one does for you. `with` guarantees the file is closed, even when the code
inside it raises; before `with` existed, that took a `try` and a `finally` around every file. `for`
works on a list, a string, a dictionary, an open file and a `csv.reader`: five different types, one
statement.

Your own classes get neither. `for reading in station:` raises `TypeError: 'Station' object is not
iterable`, and so do `sum(station)` and `max(station)`. And the atomic write from the **Writing Safely** notebook is several careful lines that every script has to copy correctly, when what you
want is `with AtomicWriter(path) as f:` and the care kept in one place.

Neither statement belongs to the built-in types. Each one calls methods, and any class that has
those methods gets the statement.

### What the two protocols are

> A **context manager** is an object with `__enter__` and `__exit__`. `with thing as name:` calls
> `thing.__enter__()`, binds what it returns to `name`, runs the block, and then calls
> `thing.__exit__(...)` whether the block finished or raised.
>
> An **iterable** is an object whose `__iter__` returns an **iterator**: an object whose
> `__next__` returns one value per call and raises `StopIteration` when there are none left.
> `for item in thing:` calls `iter(thing)` once, then `next` until `StopIteration`.

### Why it works that way

A context manager's value is the guarantee. `__exit__` runs on every way out of the block:
reaching the end, a `return`, a `break`, or an exception. That makes it the right place for anything
that must happen afterward, such as closing a file, deleting a temporary one, or putting a setting
back. It is passed a description of the exception, or three `None` values if there was none, so it
can behave differently on failure. The atomic writer below replaces the target when the block
succeeds and throws its temporary file away when it does not.

The split between iterable and iterator exists so an object can be looped over more than once. The
iterator holds the position; the iterable holds the data and hands a fresh iterator to every loop
that asks. An object that is its own iterator can be looped over exactly once, and the second loop
produces nothing and raises nothing.

Writing `__next__` by hand means tracking that position yourself. A method containing `yield` does
it for you: calling it returns a **generator**, an iterator whose position is simply the point where
the method paused. Most `__iter__` methods in real code are three lines with `yield` in them.

### Where you will meet this

Context managers: `open`, `zipfile.ZipFile`, `tempfile.TemporaryDirectory`, database connections,
and `pytest.raises` in the **Testing and Packaging** guide. Iterables: open files line by line,
`csv.reader`, `dict.items()`, `range` and `Path.iterdir`.

### What this notebook covers

Context managers first: the order of the calls, the guarantee on error, what `as` binds, what
`__exit__` returns, and then the atomic write as a class. Then iteration: what `for` does one step at
a time, the explicit protocol, what iterating gives you for free, and `yield`.

### A first look

Three lines added to a class. There is nothing to run yet: read it, and read the output underneath
it.

```python
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __iter__(self):
        for value in self.readings:
            yield value


north = Station("Tromso", [-4.1, -2.6, -3.8])
print(list(north))
print(max(north), min(north), -2.6 in north)
```

```
[-4.1, -2.6, -3.8]
-2.6 -4.1 True
```

`list`, `max`, `min` and `in` all work, and none of them was written. Each one only needs to loop.


## Setup

Four imports and a folder to work in.

- `os` provides `os.replace`, which the atomic writer uses to put its finished file in place
- `tempfile` creates the temporary file the atomic writer writes to first
- `Path` builds paths and reads the written file back
- `shutil` removes the scratch folder at the end

The folder is only used by the atomic writer. Everything else in this notebook stays in memory.

**Run this cell before the rest of the notebook.**


In [1]:
import os
import tempfile
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print("working in:", scratch)


working in: scratch


## Worked examples

### The order of the calls

A context manager that prints from both methods shows exactly when each one runs.


In [2]:
class Announce:
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        print(f"  __enter__ ({self.label})")
        return self

    def __exit__(self, exc_type, exc, tb):
        print(f"  __exit__  ({self.label}), exc_type is {exc_type}")
        return False


with Announce("first") as announcer:
    print("  inside the block, announcer is a", type(announcer).__name__)

print("after the block")


  __enter__ (first)
  inside the block, announcer is a Announce
  __exit__  (first), exc_type is None
after the block


`__enter__` first, then the block, then `__exit__`, then the line after the `with`.

`__exit__` takes three parameters after `self`. Together they describe an exception: its class,
the exception itself, and the traceback. When the block finished normally, all three are `None`,
which is what `exc_type is None` printed.

### `__exit__` runs when the block raises

This is the reason the protocol exists. The block below raises halfway through.


In [3]:
try:
    with Announce("second"):
        print("  about to raise")
        raise ValueError("bad reading")
        print("  never printed")
except ValueError as error:
    print("  caught outside the with:", error)


  __enter__ (second)
  about to raise
  __exit__  (second), exc_type is <class 'ValueError'>
  caught outside the with: bad reading


The line after the `raise` never ran, and `__exit__` still did, before the exception reached the
`except`. This time `exc_type` is `ValueError`, so the method knows the block failed.

That ordering is the whole guarantee. Whatever `__exit__` does, closing a file or removing a
temporary one, happens on the way out no matter how the block ended.

### `as` binds what `__enter__` returns

The name after `as` is not the object in the `with` line. It is whatever `__enter__` returned.


In [4]:
class GivesNumber:
    def __enter__(self):
        return 42

    def __exit__(self, exc_type, exc, tb):
        return False


with GivesNumber() as value:
    print("value is", value)


value is 42


`open` returns the file from `__enter__`, which is why `with open(...) as f:` hands you something
you can read from. Returning `self` is the common case and the convention when there is nothing
better to return, but the atomic writer below returns something else.

### What `__exit__` returns

`__exit__` returning a true value tells Python the exception has been dealt with, and it stops
there.


In [5]:
class Swallows:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return True


with Swallows():
    raise ValueError("this should have stopped the program")

print("execution continued as though nothing had happened")


execution continued as though nothing had happened


The `ValueError` was raised, reached `__exit__`, and vanished. Nothing printed an error, and the
next line ran.

Almost every context manager should return `False`, or return nothing, which gives `None` and has
the same effect. A context manager that hides every exception inside its block turns a crash with a
traceback into a program that carries on with wrong data.

### The atomic write as a context manager

The **Writing Safely** notebook wrote to a temporary file beside the target, then moved it into place
with `os.replace`, so an interrupted write never damaged the original. Here is that pattern as a
class.

`__enter__` creates the temporary file and returns the open file, so `as f` gives the block
something to write to. `__exit__` closes it first, because Windows will not replace a file that is
still open, and then does one of two things depending on how the block ended.


In [6]:
class AtomicWriter:
    def __init__(self, target):
        self.target = Path(target)

    def __enter__(self):
        handle, name = tempfile.mkstemp(dir=self.target.parent, suffix=".part")
        self.temporary = Path(name)
        self.file = open(handle, "w", encoding="utf-8")
        return self.file

    def __exit__(self, exc_type, exc, tb):
        self.file.close()
        if exc_type is None:
            os.replace(self.temporary, self.target)
        else:
            self.temporary.unlink(missing_ok=True)
        return False


target = scratch / "readings.txt"

with AtomicWriter(target) as f:
    f.write("-4.1\n-2.6\n")

print("written:", repr(target.read_text(encoding="utf-8")))


written: '-4.1\n-2.6\n'


Now the same thing, interrupted partway through the block.


In [7]:
try:
    with AtomicWriter(target) as f:
        f.write("-3.8\n")
        raise RuntimeError("interrupted")
except RuntimeError as error:
    print("interrupted:", error)

print("the file still holds:", repr(target.read_text(encoding="utf-8")))
print("leftover .part files:", list(scratch.glob("*.part")))


interrupted: interrupted
the file still holds: '-4.1\n-2.6\n'
leftover .part files: []


The target still holds the first write, and no temporary file was left behind.

The block that uses this has no idea any of it happened. It writes to `f` as it would to any file.
The care that the **Writing Safely** notebook spread across a `try`, a `finally` and an
`os.replace` now lives in one class, written once, and every caller gets it by writing `with`.

### What `for` does, one step at a time

`for` is built from two functions you can call yourself. `iter` asks an object for an iterator, and
`next` asks the iterator for one value.


In [8]:
readings = [-4.1, -2.6, -3.8]

it = iter(readings)
print("iter() gave a", type(it).__name__)

print("next:", next(it))
print("next:", next(it))
print("next:", next(it))

try:
    next(it)
except StopIteration:
    print("StopIteration, which is how a loop learns it is finished")


iter() gave a list_iterator
next: -4.1
next: -2.6
next: -3.8
StopIteration, which is how a loop learns it is finished


That is everything `for reading in readings:` does: one call to `iter`, then `next` until
`StopIteration`, which the loop catches silently.

The list did not change. `list_iterator` is a separate object holding a position in the list, and
that separation is what the next section builds.

### The explicit protocol

Two classes. `ReadingIterator` holds a position and hands out one value per `__next__`. `Station`
holds the readings, and its `__iter__` makes a new `ReadingIterator` each time it is asked.

An iterator's own `__iter__` returns `self`, because an iterator is also allowed to appear in a
`for` loop.


In [9]:
class ReadingIterator:
    def __init__(self, readings):
        self.readings = readings
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.readings):
            raise StopIteration
        value = self.readings[self.position]
        self.position += 1
        return value


class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __iter__(self):
        return ReadingIterator(self.readings)


north = Station("Tromso", [-4.1, -2.6, -3.8])

for reading in north:
    print(reading)

print("and again:", list(north))


-4.1
-2.6
-3.8
and again: [-4.1, -2.6, -3.8]


The second loop worked because it received a new `ReadingIterator`, starting from position zero.
The station itself holds no position at all.

### What iterating gives you for free

Nothing else was written. Every one of these only ever loops.


In [10]:
print("list:     ", list(north))
print("sum:      ", round(sum(north), 1))
print("max, min: ", max(north), min(north))
print("sorted:   ", sorted(north))
print("in:       ", -2.6 in north)
print("enumerate:", list(enumerate(north)))

first, second, third = north
print("unpacked: ", first, second, third)


list:      [-4.1, -2.6, -3.8]
sum:       -10.5
max, min:  -2.6 -4.1
sorted:    [-4.1, -3.8, -2.6]
in:        True
enumerate: [(0, -4.1), (1, -2.6), (2, -3.8)]
unpacked:  -4.1 -2.6 -3.8


`in` works without an `__eq__` of its own because it loops and compares each reading, and the
readings are numbers. Unpacking works because it is iteration too, which the **Tuples and Unpacking** notebook relied on without saying so.

`len(north)` would still raise. Length is a separate protocol, `__len__` from the **Dunder Methods**
notebook, because not everything you can loop over knows its length in advance.

### `yield`: the short way

`ReadingIterator` is twelve lines of bookkeeping: a position, a bounds check, an increment, a
`StopIteration`. A method containing `yield` replaces all of it.


In [11]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __iter__(self):
        for value in self.readings:
            yield value


south = Station("Malaga", [18.9, 19.4])

print("__iter__ now returns a", type(iter(south)).__name__)
print("list:    ", list(south))
print("again:   ", list(south))


__iter__ now returns a generator
list:     [18.9, 19.4]
again:    [18.9, 19.4]


`yield` makes `__iter__` return a **generator** rather than running to the end. Each `next` resumes
the method from where it last paused, runs until the next `yield`, and hands that value out. When
the method reaches its end, the generator raises `StopIteration` on its own.

The second loop worked for the same reason as before: each call to `__iter__` creates a new
generator.

### A generator runs only when asked

Nothing in a generator runs until something asks for a value. Printing from inside one shows the
order.


In [12]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        print("    (starting)")
        value = self.start
        while value > 0:
            print(f"    (about to yield {value})")
            yield value
            value -= 1
        print("    (finished)")


steps = iter(Countdown(2))
print("created, and nothing has printed yet")

print("first next: ", next(steps))
print("second next:", next(steps))

try:
    next(steps)
except StopIteration:
    print("StopIteration once the method body ended")


created, and nothing has printed yet
    (starting)
    (about to yield 2)
first next:  2
    (about to yield 1)
second next: 1
    (finished)
StopIteration once the method body ended


`iter(Countdown(2))` printed nothing. The body started on the first `next`, stopped at the first
`yield`, and did not continue until the second `next` asked it to.

This is why a generator can describe a sequence without storing it. A `Countdown(1_000_000)` holds
one number at a time, and the **Comprehensions** notebook measured the same memory difference for
generator expressions.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/05-context-managers-and-iterators-solutions.ipynb).

**1.** Write a context manager `Section(title)` that prints `== title ==` on the way in and
`== end title ==` on the way out. Use it around two `print` calls.


In [13]:
# your code here


**2.** Raise an exception inside a `Section` block and catch it outside the `with`. Show that the
closing line still printed, and say in a comment what `__exit__` returned to let the exception
through.


In [14]:
# your code here


**3.** Write `TemporaryUnit(station, unit)`, which sets the station's `unit` on the way in and puts
the previous value back on the way out. Show the unit inside and after a normal block, then show it
restored after a block that raises.


In [15]:
# your code here


**4.** Write a `Shelf` class holding a list of book titles, iterable through a separate iterator
class with `__next__`. Loop over one shelf twice to show the second loop works.


In [16]:
# your code here


**5.** Rewrite `Shelf.__iter__` using `yield`, delete the iterator class, and show `list`, `in` and a
second loop all working.


In [17]:
# your code here


**6.** Write an iterable `Countdown(start)` that yields `start`, then one less, down to `1`, with no
printing inside it. Use it in a `for` loop, then pass one to `sum`.


In [18]:
# your code here


## Common errors

### TypeError: the object does not support the context manager protocol

`with` on an object that has neither method.


In [19]:
class NotAManager:
    def __init__(self, name):
        self.name = name


with NotAManager("Tromso"):
    print("never reached")


TypeError: 'NotAManager' object does not support the context manager protocol (missed __exit__ method)

Read the end of the message carefully, because it names only one method and the class is missing
both. Python looks for `__exit__` first and reports that, so adding `__exit__` alone and running
again produces the same error naming `__enter__`. A context manager needs both.

### TypeError: `__exit__` written without its three parameters

`__exit__` is always called with three arguments after `self`, even when there was no exception.


In [20]:
class ShortExit:
    def __enter__(self):
        return self

    def __exit__(self):
        return False


with ShortExit():
    print("the block ran")


the block ran


TypeError: ShortExit.__exit__() takes 1 positional argument but 4 were given

The block ran and printed. The error arrived afterward, when Python called `__exit__` and passed it
three `None` values it had no parameters for.

That timing matters. A context manager with a broken `__exit__` lets the block do its work and then
fails to clean up after it. Write the signature as `__exit__(self, exc_type, exc, tb)` every time,
including when you ignore all three.

### TypeError: the object is not iterable

`for` on an object with no `__iter__`.


In [21]:
class NoIter:
    def __init__(self, readings):
        self.readings = readings


for reading in NoIter([-4.1, -2.6]):
    print(reading)


TypeError: 'NoIter' object is not iterable

`sum`, `max`, `list`, `sorted` and unpacking all raise the same error on this object, because each of
them calls `iter` first.

### TypeError: `__iter__` returned a list

This is the obvious first attempt: the station already holds a list, so hand it back.


In [22]:
class ReturnsList:
    def __init__(self, readings):
        self.readings = readings

    def __iter__(self):
        return self.readings


for reading in ReturnsList([-4.1, -2.6]):
    print(reading)


TypeError: iter() returned non-iterator of type 'list'

A list is iterable, but it is not an iterator. It has no `__next__`, so `for` has nothing to call.

Wrapping it in `iter()` hands back the list's own iterator, which does have one. When all `__iter__`
does is walk a list, this is the shortest correct version.


In [23]:
class ReturnsIterator:
    def __init__(self, readings):
        self.readings = readings

    def __iter__(self):
        return iter(self.readings)


print(list(ReturnsIterator([-4.1, -2.6])))


[-4.1, -2.6]


### The quiet one: an object that is its own iterator

Putting the position on the object itself, rather than on a separate iterator, looks simpler and
works the first time.


In [24]:
class OneShot:
    def __init__(self, readings):
        self.readings = readings
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.readings):
            raise StopIteration
        value = self.readings[self.position]
        self.position += 1
        return value


once = OneShot([-4.1, -2.6, -3.8])

print("first loop: ", list(once))
print("second loop:", list(once))
print("sum now:    ", sum(once))


first loop:  [-4.1, -2.6, -3.8]
second loop: []
sum now:     0


The second loop produced an empty list and `sum` returned `0`, and neither raised. An exhausted
iterator is exactly how a loop learns it is finished, so an empty result looks like a station with no
readings rather than a bug.

The first loop moved `position` to the end, and `__iter__` returned `self`, so every later loop was
handed an iterator that had already finished. The fix is for `__iter__` to return a **new** iterator
each time, which the separate iterator class does, and which `yield` does automatically.

A generator is itself single use, as the **Comprehensions** notebook showed for generator
expressions. That is why `yield` belongs inside `__iter__`, where every loop gets a fresh one, and
not in a generator stored on the object.

### Cleaning up


In [25]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- `with` calls `__enter__`, binds what it returns to the name after `as`, runs the block, then calls
  `__exit__`.
- `__exit__` runs on every way out of the block, including an exception, which makes it the place for
  cleanup.
- `__exit__` takes three parameters describing the exception, all `None` when there was none.
- Returning a true value from `__exit__` hides the exception. Return `False`, or nothing.
- `as` binds what `__enter__` returns: usually `self`, but not always.
- A context manager can keep a careful pattern, such as an atomic write, in one class.
- `for` calls `iter()` once, then `next()` until `StopIteration`.
- An iterable's `__iter__` returns a new iterator, and the iterator holds the position.
- Iterating gives you `list`, `sum`, `max`, `min`, `sorted`, `in` and unpacking without writing them.
- `__iter__` can return `iter(self.readings)` when all it does is walk a list.
- A method containing `yield` returns a generator, which tracks its own position and runs only when
  asked.
- An object that is its own iterator can be looped over exactly once, and the second loop is empty
  rather than an error.


## What is next

The **Decorators** notebook. **Properties**, **Class and Static Methods** and **Dataclasses** all
put a line beginning with `@` above a method or a class, and nothing so far has said what that line
does. It turns out to be ordinary Python that takes a function and returns one, and one of its
examples is a shorter way to write the context managers in this notebook.


---

&#8592; **Previous:** [Dunder Methods](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/04-dunder-methods.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
